# Exercício: Spark Streaming — Voos de Companhias Aéreas → PostgreSQL **LOCAL**

**Objetivo:** simular um fluxo contínuo de eventos de voos, processar com **Spark Structured Streaming** usando **watermark** para calcular a **evolução dos voos ao longo do tempo** (janelas temporais por companhia) e gravar os resultados no **PostgreSQL instalado na própria VM do Colab** (`localhost`).

| Etapa | Tecnologia |
|-------|------------|
| Fonte de dados | eventos de voos simulados (JSON) |
| Stream | pasta monitorada pelo Spark |
| Processamento | PySpark Structured Streaming + **Watermark** |
| Agregação | janelas temporais (`window`) por companhia |
| Banco | **PostgreSQL local** (`127.0.0.1:5432`) |

> ⚠️ **Runtime → Factory reset runtime** antes de executar, para garantir uma VM limpa.

### O que é *watermark*?
O **watermark** informa ao Spark **até quando** ele deve esperar por eventos atrasados (*late data*). Eventos que chegarem com um *event-time* mais antigo que o watermark atual são **descartados**, e o Spark pode **finalizar (emitir) as janelas** já completas. Isso é essencial para agregações por janela temporal em fluxos contínuos.

## 1. Instalar PostgreSQL **local** no Colab

In [ ]:
%%bash
set -e

echo "=== Instalando PostgreSQL LOCAL no Colab ==="
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql postgresql-contrib > /dev/null

service postgresql start
sleep 4

# Cria usuário, senha e banco para o Spark
sudo -u postgres psql <<'SQL'
ALTER USER postgres WITH PASSWORD 'colab_root';
DROP DATABASE IF EXISTS voos;
CREATE DATABASE voos;
DO $$
BEGIN
   IF NOT EXISTS (SELECT FROM pg_catalog.pg_roles WHERE rolname = 'spark') THEN
      CREATE ROLE spark LOGIN PASSWORD 'spark123';
   END IF;
END
$$;
GRANT ALL PRIVILEGES ON DATABASE voos TO spark;
SQL

# Garante permissões de schema no banco voos
sudo -u postgres psql -d voos <<'SQL'
GRANT ALL ON SCHEMA public TO spark;
ALTER SCHEMA public OWNER TO spark;
SQL

# Permite conexão local via senha (md5)
PG_HBA=$(sudo -u postgres psql -tAc "SHOW hba_file;")
echo "host    all    all    127.0.0.1/32    md5" >> "$PG_HBA"
echo "host    all    all    ::1/128         md5" >> "$PG_HBA"
service postgresql restart
sleep 4

PGPASSWORD=spark123 psql -h 127.0.0.1 -U spark -d voos -c "SELECT 'PostgreSQL LOCAL OK' AS status, current_database() AS banco, inet_server_port() AS porta;"

## 2. Dependências Python + configuração local

In [ ]:
%pip -q install pyspark psycopg2-binary

import json
import time
import random
import threading
from datetime import datetime, timedelta
from pathlib import Path

import psycopg2

# ============================================================
# POSTGRESQL 100% LOCAL — NÃO USA CLOUD, NÃO USA SECRETS
# ============================================================
PG_HOST = "127.0.0.1"
PG_PORT = 5432
PG_DATABASE = "voos"
PG_USER = "spark"
PG_PASSWORD = "spark123"

# Companhias aéreas e aeroportos simulados
COMPANHIAS = ["LATAM", "GOL", "AZUL", "AVIANCA", "TAP"]
AEROPORTOS = ["GRU", "GIG", "BSB", "CNF", "SSA", "REC", "POA", "CWB", "FOR", "MAO"]
STATUS_VOO = ["DECOLADO", "EM_ROTA", "POUSADO", "ATRASADO", "CANCELADO"]

STREAM_DIR = Path("/content/voos_stream")
STREAM_DIR.mkdir(parents=True, exist_ok=True)
INTERVALO_SEGUNDOS = 5
DURACAO_PRODUTOR_SEG = 120

JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
PG_CONNECTOR_JAR = "/content/postgresql-42.7.3.jar"

# Teste de conexão local
conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT,
    user=PG_USER, password=PG_PASSWORD, dbname=PG_DATABASE,
)
with conn.cursor() as cur:
    cur.execute("SELECT current_database(), inet_server_port()")
    banco, porta = cur.fetchone()
conn.close()

print("=" * 50)
print("  BANCO: PostgreSQL LOCAL (instalado neste Colab)")
print(f"  Host:  {PG_HOST}  |  Porta: {porta}  |  DB: {banco}")
print(f"  User:  {PG_USER}")
print("=" * 50)

## 3. Baixar driver JDBC e criar tabela `evolucao_voos`

Vamos gravar a **agregação por janela temporal**: quantidade de voos por companhia dentro de cada janela de tempo (a "evolução ao longo do tempo").

In [ ]:
!wget -q -O {PG_CONNECTOR_JAR} https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar

CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS evolucao_voos (
    id              BIGSERIAL PRIMARY KEY,
    companhia       VARCHAR(30)  NOT NULL,
    janela_inicio   TIMESTAMP    NOT NULL,
    janela_fim      TIMESTAMP    NOT NULL,
    total_voos      BIGINT       NOT NULL,
    passageiros     BIGINT,
    atraso_medio_min DOUBLE PRECISION,
    processado_em   TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);
CREATE INDEX IF NOT EXISTS idx_evolucao_comp_janela
    ON evolucao_voos (companhia, janela_inicio);
"""

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT,
    user=PG_USER, password=PG_PASSWORD, dbname=PG_DATABASE,
)
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_TABLE_SQL)
    conn.commit()
    print("✅ Tabela `evolucao_voos` pronta no PostgreSQL local.")
finally:
    conn.close()

## 4. Produtor — simula stream de eventos de voos

Em produção a fonte seria Kafka, um WebSocket da malha aérea ou uma API (ex.: OpenSky). Aqui gravamos JSONs em disco para o Spark consumir. Cada evento tem um **`event_time`** — o horário real do evento — que será usado pelo watermark.

In [ ]:
def gerar_evento_voo() -> dict:
    """Gera um evento de voo sintético com event-time."""
    origem, destino = random.sample(AEROPORTOS, 2)
    companhia = random.choice(COMPANHIAS)
    numero = random.randint(1000, 9999)

    # Simula late data: às vezes o evento chega com atraso de alguns segundos
    atraso_evento = random.choice([0, 0, 0, 2, 5, 8])
    event_time = datetime.utcnow() - timedelta(seconds=atraso_evento)

    return {
        "voo": f"{companhia[:2].upper()}{numero}",
        "companhia": companhia,
        "origem": origem,
        "destino": destino,
        "status": random.choice(STATUS_VOO),
        "passageiros": random.randint(50, 300),
        "atraso_min": round(random.uniform(0, 90), 1),
        "event_time": event_time.strftime("%Y-%m-%d %H:%M:%S"),
    }


def produzir_stream(stop_event: threading.Event):
    """Grava um arquivo JSON com um lote de voos a cada intervalo."""
    inicio = time.time()
    lote = 0
    while not stop_event.is_set() and (time.time() - inicio) < DURACAO_PRODUTOR_SEG:
        lote += 1
        ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
        eventos = [gerar_evento_voo() for _ in range(random.randint(5, 15))]
        arquivo = STREAM_DIR / f"lote{lote}_{ts}.json"
        # JSON Lines: um evento por linha (formato lido pelo Spark)
        arquivo.write_text(
            "\n".join(json.dumps(e) for e in eventos), encoding="utf-8"
        )
        print(f"📥 {arquivo.name} → {len(eventos)} voos")
        time.sleep(INTERVALO_SEGUNDOS)
    print("🏁 Produtor encerrado.")


# Teste rápido de um evento
print("Amostra:", json.dumps(gerar_evento_voo(), indent=2, ensure_ascii=False))

## 5. Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, window, count, sum as _sum, avg
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType,
)

spark = (
    SparkSession.builder
    .appName("VoosStreamingPostgreSQL")
    .config("spark.jars", PG_CONNECTOR_JAR)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

## 6. Structured Streaming com **Watermark** — evolução dos voos ao longo do tempo

Aqui está o coração do exercício:

1. Lemos os eventos e convertemos `event_time` em `timestamp`.
2. Aplicamos **`.withWatermark("event_time", "20 seconds")`** — o Spark tolera eventos atrasados em até 20s.
3. Agrupamos por **janela temporal de 30 segundos** e por **companhia**.
4. Calculamos o **total de voos**, **passageiros** e **atraso médio** por janela.
5. Gravamos cada janela finalizada no PostgreSQL via `foreachBatch`.

In [ ]:
schema = StructType([
    StructField("voo", StringType(), True),
    StructField("companhia", StringType(), False),
    StructField("origem", StringType(), True),
    StructField("destino", StringType(), True),
    StructField("status", StringType(), True),
    StructField("passageiros", IntegerType(), True),
    StructField("atraso_min", DoubleType(), True),
    StructField("event_time", StringType(), False),
])

# Leitura do stream (arquivos JSON Lines na pasta monitorada)
eventos_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 5)
    .json(str(STREAM_DIR))
    .withColumn("event_time", to_timestamp(col("event_time")))
)

# Agregação por janela temporal com WATERMARK
evolucao_df = (
    eventos_df
    .withWatermark("event_time", "20 seconds")
    .groupBy(
        window(col("event_time"), "30 seconds"),
        col("companhia"),
    )
    .agg(
        count("*").alias("total_voos"),
        _sum("passageiros").alias("passageiros"),
        avg("atraso_min").alias("atraso_medio_min"),
    )
    .select(
        col("companhia"),
        col("window.start").alias("janela_inicio"),
        col("window.end").alias("janela_fim"),
        col("total_voos"),
        col("passageiros"),
        col("atraso_medio_min"),
    )
)


def escrever_postgres(batch_df, batch_id: int):
    """Callback foreachBatch: grava as janelas agregadas no PostgreSQL."""
    if batch_df.isEmpty():
        return

    qtd = batch_df.count()
    print(f"Batch {batch_id}: gravando {qtd} janela(s) agregada(s) no PostgreSQL...")
    batch_df.orderBy("janela_inicio", "companhia").show(truncate=False)

    (
        batch_df.write
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", "evolucao_voos")
        .option("user", PG_USER)
        .option("password", PG_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .mode("append")
        .save()
    )


# Output mode "append" só emite janelas já finalizadas pelo watermark
query = (
    evolucao_df.writeStream
    .outputMode("append")
    .foreachBatch(escrever_postgres)
    .option("checkpointLocation", "/content/checkpoint_voos")
    .trigger(processingTime="15 seconds")
    .start()
)

print(f"Streaming query id: {query.id}")

## 7. Executar produtor + aguardar streaming

O produtor roda em background gerando lotes de voos. O Spark agrega por janela e, conforme o **watermark avança**, grava as janelas completas no PostgreSQL.

In [ ]:
stop_event = threading.Event()
produtor = threading.Thread(target=produzir_stream, args=(stop_event,), daemon=True)
produtor.start()

# Aguarda produtor + margem para o Spark fechar as últimas janelas (watermark)
try:
    tempo_total = DURACAO_PRODUTOR_SEG + 60
    print(f"⏳ Aguardando ~{tempo_total}s (produção + fechamento das janelas)...")
    query.awaitTermination(timeout=tempo_total)
finally:
    stop_event.set()
    produtor.join(timeout=10)
    query.stop()
    print("✅ Streaming finalizado.")

## 8. Consultar a evolução dos voos gravada no PostgreSQL

In [ ]:
import pandas as pd

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT,
    user=PG_USER, password=PG_PASSWORD, dbname=PG_DATABASE,
)
try:
    df = pd.read_sql(
        """
        SELECT companhia, janela_inicio, janela_fim,
               total_voos, passageiros, ROUND(atraso_medio_min::numeric, 1) AS atraso_medio_min
        FROM evolucao_voos
        ORDER BY janela_inicio, companhia
        """,
        conn,
    )
finally:
    conn.close()

print(f"Total de janelas agregadas gravadas: {len(df)}")
df

## 9. Visualizar a evolução dos voos ao longo do tempo

Gráfico do total de voos por janela temporal, por companhia.

In [ ]:
import matplotlib.pyplot as plt

if len(df) == 0:
    print("Nenhum dado para plotar. Rode as células anteriores novamente.")
else:
    fig, ax = plt.subplots(figsize=(11, 5))
    for companhia, grupo in df.groupby("companhia"):
        grupo = grupo.sort_values("janela_inicio")
        ax.plot(grupo["janela_inicio"], grupo["total_voos"],
                marker="o", label=companhia)

    ax.set_title("Evolução dos voos ao longo do tempo (janelas de 30s)")
    ax.set_xlabel("Início da janela (event-time)")
    ax.set_ylabel("Total de voos")
    ax.legend(title="Companhia")
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 10. (Opcional) Encerrar Spark

Libera os recursos da sessão Spark.

In [ ]:
try:
    query.stop()
except Exception:
    pass
spark.stop()
print("🛑 Spark encerrado.")